In [1]:
import requests
import pandas as pd
import json
import psycopg
import os

from dotenv import load_dotenv

In [2]:
load_dotenv()
print(os.getenv("DB_NAME"))

proj1_weather_api


In [3]:
url = "https://archive-api.open-meteo.com/v1/archive"

In [4]:
locs = [
    ("Fresno County", 36.7378, -119.7871),
    ("Kern County", 35.3733, -119.0187),
    ("Tulare County", 36.3302, -119.2921),
    ("Monterey County", 36.6777, -121.6555),
    ("Merced County", 37.3022, -120.4829),
    ("Stanislaus County", 37.6391, -120.9969),
    ("San Joaquin County", 37.9577, -121.2908),
    ("Ventura County", 34.2746, -119.2290),
    ("Kings County", 36.3275, -119.6457),
    ("Imperial County", 32.7920, -115.5631),
    ("San Diego County", 32.7157, -117.1611),
    ("Madera County", 36.9613, -120.0607),
    ("San Luis Obispo County", 35.2828, -120.6596),
    ("Santa Barbara County", 34.9530, -120.4357),
    ("Yolo County", 38.6785, -121.7733),
    ("Colusa County", 39.2143, -122.0094),
    ("Butte County", 39.5138, -121.5564),
    ("Glenn County", 39.5243, -122.1936),
    ("Sutter County", 39.1404, -121.6169),
    ("Tehama County", 40.1785, -122.2358)
]

weather_df = pd.DataFrame()
all_counties = []

for county, lat, long in locs:
    params = {
        "latitude": lat,
        "longitude": long,
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "daily": 
            "temperature_2m_mean,"
            "et0_fao_evapotranspiration_sum,"
            "soil_moisture_0_to_100cm_mean,"
            "soil_temperature_0_to_100cm_mean,"
            "relative_humidity_2m_mean,"
            "vapour_pressure_deficit_max,"
            "wind_speed_10m_mean,"
            "cloud_cover_mean,"
            "precipitation_sum"
    }
    response = requests.get(url, params=params)
    data = response.json()
    data["county"] = county
    data["latitude"] = lat
    data["longitude"] = long
    all_counties.append(data)

    df = pd.DataFrame(data["daily"])
    df["time"] = pd.to_datetime(df["time"])
    df["county"] = county
    df["latitude"] = lat
    df["longitude"] = long

    weather_df = pd.concat([weather_df, df], ignore_index=True)

with open("all_counties_weather_data.json", "w") as file:
    json.dump(all_counties, file, indent=4)

weather_df = weather_df[
    [
        "county",
        "time",
        "latitude",
        "longitude",
        "temperature_2m_mean",
        "et0_fao_evapotranspiration_sum",
        "soil_moisture_0_to_100cm_mean",
        "soil_temperature_0_to_100cm_mean",
        "relative_humidity_2m_mean",
        "vapour_pressure_deficit_max",
        "wind_speed_10m_mean",
        "cloud_cover_mean",
        "precipitation_sum"
    ]
]

In [5]:
weather_df

,county,time,latitude,longitude,temperature_2m_mean,et0_fao_evapotranspiration_sum,soil_moisture_0_to_100cm_mean,soil_temperature_0_to_100cm_mean,relative_humidity_2m_mean,vapour_pressure_deficit_max,wind_speed_10m_mean,cloud_cover_mean,precipitation_sum
0,Fresno County,2024-01-01,36.7378,-119.7871,9.6,1.07,0.197,14.1,87,0.46,5.4,70,0.0
1,Fresno County,2024-01-02,36.7378,-119.7871,7.8,0.95,0.195,13.8,88,0.52,6.4,46,0.0
2,Fresno County,2024-01-03,36.7378,-119.7871,9.7,1.15,0.201,13.6,81,0.47,9.0,72,7.8
3,Fresno County,2024-01-04,36.7378,-119.7871,6.8,1.09,0.203,13.3,87,0.59,6.2,30,0.0
4,Fresno County,2024-01-05,36.7378,-119.7871,6.7,1.34,0.202,13.1,77,0.65,5.9,38,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7315,Tehama County,2024-12-27,40.1785,-122.2358,11.8,0.82,0.364,11.9,89,0.29,28.4,100,7.6
7316,Tehama County,2024-12-28,40.1785,-122.2358,13.2,0.79,0.366,12.3,88,0.33,25.3,99,7.0
7317,Tehama County,2024-12-29,40.1785,-122.2358,14.2,1.47,0.368,12.8,84,0.80,28.6,70,13.9
7318,Tehama County,2024-12-30,40.1785,-122.2358,8.8,1.73,0.370,12.4,68,1.02,13.5,1,0.0


In [6]:
weather_df.isnull().sum()

county                              0
time                                0
latitude                            0
longitude                           0
temperature_2m_mean                 0
et0_fao_evapotranspiration_sum      0
soil_moisture_0_to_100cm_mean       0
soil_temperature_0_to_100cm_mean    0
relative_humidity_2m_mean           0
vapour_pressure_deficit_max         0
wind_speed_10m_mean                 0
cloud_cover_mean                    0
precipitation_sum                   0
dtype: int64

In [7]:
weather_df.dtypes

county                                         str
time                                datetime64[us]
latitude                                   float64
longitude                                  float64
temperature_2m_mean                        float64
et0_fao_evapotranspiration_sum             float64
soil_moisture_0_to_100cm_mean              float64
soil_temperature_0_to_100cm_mean           float64
relative_humidity_2m_mean                    int64
vapour_pressure_deficit_max                float64
wind_speed_10m_mean                        float64
cloud_cover_mean                             int64
precipitation_sum                          float64
dtype: object

In [8]:
weather_df.describe()

,time,latitude,longitude,temperature_2m_mean,et0_fao_evapotranspiration_sum,soil_moisture_0_to_100cm_mean,soil_temperature_0_to_100cm_mean,relative_humidity_2m_mean,vapour_pressure_deficit_max,wind_speed_10m_mean,cloud_cover_mean,precipitation_sum
count,7320,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000,7320.000000
mean,2024-07-01 12:00:00,36.878735,-120.333215,18.148661,4.143161,0.214232,19.920683,58.636475,2.833761,8.619126,36.002869,1.614344
min,2024-01-01 00:00:00,32.715700,-122.235800,2.900000,0.290000,0.066000,8.700000,10.000000,0.170000,1.200000,0.000000,0.000000
25%,2024-04-01 00:00:00,35.350675,-121.626550,11.600000,1.920000,0.155000,13.300000,37.000000,0.880000,6.000000,2.000000,0.000000
50%,2024-07-01 12:00:00,36.849550,-120.571250,16.200000,3.780000,0.190000,18.500000,65.000000,1.750000,7.700000,29.000000,0.000000
75%,2024-10-01 00:00:00,38.793975,-119.557300,24.400000,6.150000,0.267000,26.300000,80.000000,4.590000,10.300000,65.000000,0.000000
max,2024-12-31 00:00:00,40.178500,-115.563100,40.400000,13.470000,0.435000,38.200000,97.000000,12.620000,36.700000,100.000000,82.000000
std,NaN,2.128164,1.674126,8.072456,2.493479,0.078733,7.040500,23.394160,2.472357,3.958100,33.479767,5.520378


In [9]:
weather_df.duplicated().sum()

np.int64(0)

In [10]:
# weather_df["month"] = weather_df["time"].dt.month_name()
# weather_df = weather_df.drop(columns=["time"])

# weather_summary = weather_df.groupby(["county", "latitude", "longitude", "month"],as_index=False).mean(numeric_only=True)
# months = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
# weather_summary["month"] = pd.Categorical(weather_summary["month"],categories=months,ordered=True)
# weather_summary = weather_summary.sort_values(["county", "month"])

In [11]:
weather_df

,county,time,latitude,longitude,temperature_2m_mean,et0_fao_evapotranspiration_sum,soil_moisture_0_to_100cm_mean,soil_temperature_0_to_100cm_mean,relative_humidity_2m_mean,vapour_pressure_deficit_max,wind_speed_10m_mean,cloud_cover_mean,precipitation_sum
0,Fresno County,2024-01-01,36.7378,-119.7871,9.6,1.07,0.197,14.1,87,0.46,5.4,70,0.0
1,Fresno County,2024-01-02,36.7378,-119.7871,7.8,0.95,0.195,13.8,88,0.52,6.4,46,0.0
2,Fresno County,2024-01-03,36.7378,-119.7871,9.7,1.15,0.201,13.6,81,0.47,9.0,72,7.8
3,Fresno County,2024-01-04,36.7378,-119.7871,6.8,1.09,0.203,13.3,87,0.59,6.2,30,0.0
4,Fresno County,2024-01-05,36.7378,-119.7871,6.7,1.34,0.202,13.1,77,0.65,5.9,38,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7315,Tehama County,2024-12-27,40.1785,-122.2358,11.8,0.82,0.364,11.9,89,0.29,28.4,100,7.6
7316,Tehama County,2024-12-28,40.1785,-122.2358,13.2,0.79,0.366,12.3,88,0.33,25.3,99,7.0
7317,Tehama County,2024-12-29,40.1785,-122.2358,14.2,1.47,0.368,12.8,84,0.80,28.6,70,13.9
7318,Tehama County,2024-12-30,40.1785,-122.2358,8.8,1.73,0.370,12.4,68,1.02,13.5,1,0.0


In [12]:
# query 1: Which county has the highest average wind speed in the entire year?
def query_one(cur):
    cur.execute("""
    SELECT C.county_name, AVG(W.wind_speed_10m_mean) AS average_wind_speed
    FROM County C
    LEFT JOIN Weather W ON W.county_id = C.county_id
    GROUP BY C.county_name
    ORDER BY average_wind_speed DESC
    LIMIT 1;
    """)
    return cur.fetchone()

# query 2: How many counties have moderate cloud cover (30-60%) in the entire year?
def query_two(cur):
    cur.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT C.county_name, AVG(W.cloud_cover_mean) AS average_cloud
        FROM County C
        LEFT JOIN Weather W ON W.county_id = C.county_id
        GROUP BY C.county_name
        HAVING AVG(W.cloud_cover_mean) BETWEEN 30 AND 60
        ORDER BY average_cloud DESC
    );
    """)
    return cur.fetchall()

# query 3: Which county has the highest total of precipitation in the entire year?
def query_three(cur):
    cur.execute("""
    SELECT C.county_name, SUM(W.precipitation_sum) AS total_precipitation
    FROM County C
    LEFT JOIN Weather W ON W.county_id = C.county_id
    GROUP BY C.county_name
    ORDER BY total_precipitation DESC
    LIMIT 1;
    """)
    return cur.fetchone()

# query 4: Which month has the lowest average vapour pressure deficit for every county?
def query_four(cur):
    cur.execute("""
    SELECT county_name, month, average_vapour
    FROM (
        SELECT 
            C.county_name,
            TO_CHAR(W.weather_time, 'Month') AS month,
            W.vapour_pressure_deficit_max AS average_vapour,
            ROW_NUMBER() OVER (
                PARTITION BY C.county_name ORDER BY W.vapour_pressure_deficit_max ASC
            ) as rn
        FROM County C
        LEFT JOIN Weather W ON W.county_id = C.county_id
    ) AS ranked
    WHERE rn = 1
    ORDER BY county_name;
    """)
    return cur.fetchall()

# query 5: Which 5 counties have the lowest total ET0 in the entire year?
def query_five(cur):
    cur.execute("""
    SELECT C.county_name, SUM(W.et0_fao_evapotranspiration_sum) as yearly_et0
    FROM County C
    LEFT JOIN Weather W ON W.county_id = C.county_id
    GROUP BY C.county_name
    ORDER BY yearly_et0 ASC
    LIMIT 5;
    """)
    return cur.fetchall()
    
# query 6: Which counties have ideal temperatures (18-22 Celcius)?
def query_six(cur):
    cur.execute("""
    SELECT C.county_name, AVG(W.temperature_2m_mean) as average_temp
    FROM County C
    LEFT JOIN Weather W ON W.county_id = C.county_id
    GROUP BY C.county_name
    HAVING AVG(W.temperature_2m_mean) BETWEEN 18 AND 22
    ORDER BY average_temp DESC;
    """)
    return cur.fetchall()

# query 7: What counties have a moderate average soil moisture, soil temperature, and humidity in the entire year?
def query_seven(cur):
    cur.execute("""
    WITH Averages AS (
        SELECT
            C.county_name,
            AVG(W.soil_moisture_0_to_100cm_mean) AS average_soil_moisture,
            AVG(W.soil_temperature_0_to_100cm_mean) AS average_soil_temp,
            AVG(W.relative_humidity_2m_mean) AS average_humidity
        FROM County C
        LEFT JOIN Weather W ON W.county_id = C.county_id
        GROUP BY C.county_name
    )
    SELECT *
    FROM Averages
    WHERE average_soil_moisture BETWEEN 0.2 AND 0.35
        AND average_soil_temp BETWEEN 15 AND 25
        AND average_humidity BETWEEN 60 AND 70
    LIMIT 5;
    """)
    return cur.fetchall()

In [13]:
with psycopg.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
) as conn:
    with conn.cursor() as cur:
        # create table
        cur.execute("""
        DROP TABLE IF EXISTS Weather;
        DROP TABLE IF EXISTS County;
        
        CREATE TABLE IF NOT EXISTS County (
            county_id SERIAL PRIMARY KEY,
            county_name VARCHAR(50),
            latitude DOUBLE PRECISION,
            longitude DOUBLE PRECISION
        );

        CREATE TABLE IF NOT EXISTS Weather (
            weather_id SERIAL PRIMARY KEY,
            county_id INTEGER REFERENCES County(county_id),
            weather_time DATE,
            temperature_2m_mean DOUBLE PRECISION,
            et0_fao_evapotranspiration_sum DOUBLE PRECISION,
            soil_moisture_0_to_100cm_mean DOUBLE PRECISION,
            soil_temperature_0_to_100cm_mean DOUBLE PRECISION,
            relative_humidity_2m_mean DOUBLE PRECISION,
            vapour_pressure_deficit_max DOUBLE PRECISION,
            wind_speed_10m_mean DOUBLE PRECISION,
            cloud_cover_mean DOUBLE PRECISION,
            precipitation_sum DOUBLE PRECISION
        );
        """)

        # add every dataframe row into table
        county_df = weather_df[["county", "latitude", "longitude"]].drop_duplicates()
        county_rows = list(county_df.itertuples(index=False, name=None))
        cur.executemany("""
        INSERT INTO County (
            county_name,
            latitude,
            longitude
        )
        VALUES (%s,%s,%s);
        """, county_rows)

        cur.execute("SELECT COUNT(*) FROM County;")
        count = cur.fetchone()[0]
        print(f"Rows inserted: {count}")

        weather_rows = []
        for row in weather_df.itertuples(index=False):
            weather_rows.append((
                row.time,
                row.temperature_2m_mean,
                row.et0_fao_evapotranspiration_sum,
                row.soil_moisture_0_to_100cm_mean,
                row.soil_temperature_0_to_100cm_mean,
                row.relative_humidity_2m_mean,
                row.vapour_pressure_deficit_max,
                row.wind_speed_10m_mean,
                row.cloud_cover_mean,
                row.precipitation_sum,
                row.county
            ))

        cur.executemany("""
        INSERT INTO Weather (
            county_id,
            weather_time,
            temperature_2m_mean,
            et0_fao_evapotranspiration_sum,
            soil_moisture_0_to_100cm_mean,
            soil_temperature_0_to_100cm_mean,
            relative_humidity_2m_mean,
            vapour_pressure_deficit_max,
            wind_speed_10m_mean,
            cloud_cover_mean,
            precipitation_sum
        )
        SELECT
            c.county_id,
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
        FROM County c
        WHERE c.county_name = %s;
        """, weather_rows)
        cur.execute("SELECT COUNT(*) FROM Weather;")
        count = cur.fetchone()[0]
        print(f"Rows inserted: {count}")

        # SQL queries
        print(query_one(cur))
        print(query_two(cur))
        print(query_three(cur))     
        print(query_four(cur))
        print(query_five(cur))
        print(query_six(cur))
        print(query_seven(cur))

Rows inserted: 20
Rows inserted: 7320
('Tehama County', 10.912841530054644)
[(17,)]
('Butte County', 1211.7)
[('Butte County', 'November ', 0.2), ('Colusa County', 'December ', 0.24), ('Fresno County', 'December ', 0.27), ('Glenn County', 'November ', 0.17), ('Imperial County', 'January  ', 0.94), ('Kern County', 'December ', 0.35), ('Kings County', 'December ', 0.39), ('Madera County', 'December ', 0.31), ('Merced County', 'December ', 0.35), ('Monterey County', 'January  ', 0.27), ('San Diego County', 'January  ', 0.23), ('San Joaquin County', 'December ', 0.24), ('San Luis Obispo County', 'February ', 0.22), ('Santa Barbara County', 'March    ', 0.37), ('Stanislaus County', 'December ', 0.24), ('Sutter County', 'December ', 0.21), ('Tehama County', 'December ', 0.19), ('Tulare County', 'November ', 0.24), ('Ventura County', 'February ', 0.2), ('Yolo County', 'December ', 0.17)]
[('Monterey County', 1129.820000000001), ('Ventura County', 1130.0799999999992), ('San Diego County', 1208

For this project, I focused on cities that were the top 20 in California for farming. I added the following daily columns: county, latitude, longitude, time,temperature_2m_mean, et0_fao_evapotranspiration_sum, soil_moisture_0_to_100cm_mean, soil_temperature_0_to_100cm_mean, r, elative_humidity_2m_mean, vapour_pressure_deficit_max, wind_speed_10m_mean, cloud_cover_mean, and precipitation_sum.

With this data, I was able to answer several questions pertaining to farming and weather conditions in California.
1. Which county has the highest average wind speed in the entire year? 
        Tehama County has the highest average wind speed with a speed of 10.91 km/h.

2. How many counties have moderate cloud cover (30-60%) in the entire year? 
        17 counties have moderate cloud cover between 30-60%.

3. Which county has the highest total of precipitation in the entire year? 
        Butte County has the highest total amount of precipitation in the entire year with 1211.7 mm.

4. Which month has the lowest average vapour pressure deficit for every county?
    - Butte County: November, 0.2
    - Colusa County: December, 0.24
    - Fresno County: December, 0.27
    - Glenn County: November, 0.17
    - Imperial County: January, 0.94
    - Kern County: December, 0.35
    - Kings County: December, 0.39
    - Madera County: December, 0.31
    - Merced County: December, 0.35
    - Monterey County: January, 0.27
    - San Diego County: January, 0.23
    - San Joaquin County: December, 0.24
    - San Luis Obispo County: February, 0.22
    - Santa Barbara County: March, 0.37
    - Stanislaus County: December, 0.24
    - Sutter County: December, 0.21
    - Tehama County: December, 0.19
    - Tulare County: November, 0.24
    - Ventura County: February, 0.2
    - Yolo County: December, 0.17

5. Which 5 counties have the lowest total ET0 in the entire year?
        Monterey County: 1129.820000000001
        Ventura County: 1130.0799999999992
        San Diego County: 1208.6000000000013
        Santa Barbara County: 1240.9699999999996
        San Luis Obispo County: 1254.27

6. Which counties have ideal temperatures (18-22 Celcius)?
        Kern County, Kings County, Tulare County, Fresno County, Glenn County, Colusa County, Madera County, Butte County, Merced County, Stanislaus County, Tehama County, and Sutter County have ideal temperatures between 18-22 degrees Celcius.

7. What counties have a moderate average soil moisture, soil temperature, and humidity in the entire year?
        Only San Joaquin County has moderate averages in all three categories for the entire year.